# 10 - P00 vs P01 vs P02

Comparacion estilo Taller 1 de los preprocesamientos del Proyecto 2.

La comparacion no reentrena modelos pesados: consolida evidencia ya medida en `investigation/` y `investigation/results/`, valida que los artefactos existan y deja una tabla unica para decidir que preprocesamientos pasan a entrenamiento/final.


## Imports y configuracion


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import zipfile

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

ROOT = Path.cwd()
if ROOT.name == '02_preprocesamiento':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
RESULTS_DIR = ROOT / '02_preprocesamiento' / 'results'
INVESTIGATION = ROOT / 'investigation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(INVESTIGATION))

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 220)

from scripts.fat2019.data import CORRUPT_OR_BAD_LABEL_FILES, split_labels
from scripts.fat2019.features import read_wav_mono, log_mel_spectrogram, extract_log_mel_stats


## Comparacion final


In [2]:
pipelines = pd.DataFrame([
    {'pipeline': 'P00_logmel_stats_basic', 'artifact': 'data/curated_logmel_stats_all.npz', 'model_family': 'LogisticRegression One-vs-Rest', 'best_private_lb': 0.37607, 'decision': 'keep', 'reason': 'baseline tabular barato y diverso'},
    {'pipeline': 'P01_logmel_image_512', 'artifact': 'data/curated_logmel_image_m128_f512.npz', 'model_family': 'CNN log-mel', 'best_private_lb': 0.52257, 'decision': 'keep', 'reason': 'salto por estructura tiempo-frecuencia'},
    {'pipeline': 'P01_plus_training_fashion', 'artifact': 'investigation/submissions/logmel_cnn_fashion_he_plateau_head256/small_logmel_cnn.csv', 'model_family': 'CNN head256 + He + plateau', 'best_private_lb': 0.64665, 'decision': 'keep', 'reason': 'mejor entrenamiento/cabeza y ensemble'},
    {'pipeline': 'P02_globalmel', 'artifact': 'data/curated_logmel_image_m128_f512_globalmel.npz', 'model_family': 'separable temporal', 'best_private_lb': 0.66561, 'decision': 'keep', 'reason': 'normalizacion global por banda mel'},
    {'pipeline': 'P02_globalmel_f1024', 'artifact': 'data/curated_logmel_image_m128_f1024.npz', 'model_family': 'separable temporal 1024', 'best_private_lb': 0.67025, 'decision': 'selected_final', 'reason': 'mayor contexto temporal + mejor blend'},
])
pipelines['artifact_exists'] = pipelines['artifact'].map(lambda p: (ROOT / p).exists())
pipelines['delta_vs_previous'] = pipelines['best_private_lb'].diff()
comparison = pipelines.sort_values('best_private_lb', ascending=False).reset_index(drop=True)
display(comparison)

out = RESULTS_DIR / '10_p00vsp01vsp02_comparison.csv'
comparison.to_csv(out, index=False)
print(f'Resultados guardados en: {out}')


,pipeline,artifact,model_family,best_private_lb,decision,reason,artifact_exists,delta_vs_previous
0,P02_globalmel_f1024,data/curated_logmel_image_m128_f1024.npz,separable temporal 1024,0.67025,selected_final,mayor contexto temporal + mejor blend,True,0.00464
1,P02_globalmel,data/curated_logmel_image_m128_f512_globalmel.npz,separable temporal,0.66561,keep,normalizacion global por banda mel,True,0.01896
2,P01_plus_training_fashion,investigation/submissions/logmel_cnn_fashion_h...,CNN head256 + He + plateau,0.64665,keep,mejor entrenamiento/cabeza y ensemble,True,0.12408
3,P01_logmel_image_512,data/curated_logmel_image_m128_f512.npz,CNN log-mel,0.52257,keep,salto por estructura tiempo-frecuencia,True,0.14650
4,P00_logmel_stats_basic,data/curated_logmel_stats_all.npz,LogisticRegression One-vs-Rest,0.37607,keep,baseline tabular barato y diverso,True,NaN


Resultados guardados en: /home/santig14/fing/taa/2_TallerAA/02_preprocesamiento/results/10_p00vsp01vsp02_comparison.csv


## Lectura rapida

- `P00` no gana, pero se conserva porque es barato y diverso.
- `P01` es el salto conceptual grande: pasar de stats a imagen log-mel.
- La etapa de entrenamiento sobre `P01` explica la zona `0.63-0.64`.
- `P02` agrega las mejoras nuevas validas: normalizacion global mel y `frames=1024`.
- El pipeline final usa `P02_globalmel_f1024` dentro de un ensemble, validado por private LB `0.67025`.
